# Pending paper bets

Currently-open bets from the most recent scan. Read-only — re-run after every `python -m src.paper_trader --scan-only`.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PENDING = REPO / 'data' / 'paper_trades' / 'pending.csv'

df = pd.read_csv(PENDING) if PENDING.exists() else pd.DataFrame()

if df.empty:
    print('No pending bets. Run `python -m src.paper_trader --scan-only` to generate some.')
else:
    print(f'n_pending      : {len(df)}')
    print(f'total exposure : ${df["entry_price"].sum():.2f}')
    print(f'avg edge       : {df["edge"].mean():+.3f}')
    print(f'min/max edge   : {df["edge"].min():+.3f} / {df["edge"].max():+.3f}')
    print(f'latest scan    : {df["timestamp_recorded"].max()}')

## Bets — sorted by edge

In [ ]:
def show_table(df):
    if df.empty:
        return df
    cols = ['chosen_player_name', 'player_a', 'player_b', 'tournament', 'surface',
            'chosen_direction', 'entry_price', 'theo_chosen', 'edge', 'kalshi_url']
    out = (df[cols]
             .sort_values('edge', ascending=False)
             .reset_index(drop=True)
             .rename(columns={
                 'chosen_player_name': 'bet_on',
                 'chosen_direction':   'side',
                 'entry_price':        'cost',
                 'theo_chosen':        'theo',
             }))
    return (out.style
             .background_gradient(subset=['edge'], cmap='Greens')
             .format({'cost': '{:.2f}', 'theo': '{:.3f}', 'edge': '{:+.3f}'}))

show_table(df)

## Why each bet was made

Top 5 features driving each bet. Green = pushed toward our bet winning; red = pushed against (so the model bet *despite* this feature).

In [ ]:
def plot_shifts(df, top_k=5):
    if df.empty:
        print('No pending bets.'); return
    sorted_df = df.sort_values('edge', ascending=False).reset_index(drop=True)
    n = len(sorted_df)
    fig, axes = plt.subplots(n, 1, figsize=(8, 1.6 * n))
    if n == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sorted_df.iterrows()):
        try:
            shifts = json.loads(row['feature_shifts_json'])
        except Exception:
            ax.set_visible(False); continue
        items = sorted(shifts.items(), key=lambda kv: -abs(kv[1]))[:top_k]
        names = [k for k, _ in items][::-1]
        vals  = [v for _, v in items][::-1]
        colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in vals]
        ax.barh(names, vals, color=colors)
        ax.axvline(0, color='black', lw=0.5)
        ax.set_title(
            f"{row['chosen_player_name']} ({row['chosen_direction']})  "
            f"theo={row['theo_chosen']:.2f}  cost={row['entry_price']:.2f}  "
            f"edge={row['edge']:+.3f}",
            fontsize=9, loc='left')
        ax.tick_params(labelsize=8)
        ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_shifts(df)